# 04 | Empirical retrieval with spatial and temporal holdouts
Fit five candidate models on 2024 training rows. A mean baseline and seasonal harmonic ridge quantify how much skill comes from seasonality. Compare SAR-only ridge, NDVI-only ridge, SAR–NDVI ridge fusion, and histogram gradient boosting fusion. Standardization is fitted only on training data. Candidate selection uses validation RMSE; all held-out metrics are reported regardless of success.

The target is SMAP monthly surface volumetric soil moisture, not in-situ soil moisture or root-zone moisture. Adjacent SMAP grid cells and repeated months are correlated; effective sample sizes are smaller than row counts.

## Setup (CPU, standard RAM)
Run this cell first in each notebook. The project uses small monthly raster tiles and tabular ML. If Colab requests a restart, use Runtime → Restart session and rerun from the beginning. Do not change NumPy manually. Files persist in Drive.

In [ ]:
%pip -q install earthengine-api rasterio pyproj scikit-learn joblib
import sys,json
from pathlib import Path
from importlib.metadata import version
for pkg,mod in [('numpy','numpy'),('scikit-learn','sklearn'),('rasterio','rasterio')]:
    loaded=sys.modules.get(mod)
    if loaded is not None and getattr(loaded,'__version__',version(pkg))!=version(pkg):
        raise RuntimeError(f'{pkg} changed. Restart session and rerun.')
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Bagre_SAR_Soil_Moisture')
assert (ROOT/'code'/'sar_workflow.py').is_file(), 'Check Drive folder and shared code.'
sys.path.insert(0,str(ROOT/'code'))
import sar_workflow as sw
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
for sub in ['data','models','outputs','figures','qgis']: (ROOT/sub).mkdir(exist_ok=True)
print('NumPy:',version('numpy'),'scikit-learn:',version('scikit-learn'))

In [ ]:
df=pd.read_csv(ROOT/'data'/'coarse_monthly_table.csv')
assert df.eligible.dtype==bool and df.core.dtype==bool
models,scores,pred,selected=sw.fit_models(df)
print('Selected on validation only:',selected)
display(scores.round(4))
scores.to_csv(ROOT/'outputs'/'model_metrics.csv',index=False)
pred.to_csv(ROOT/'outputs'/'heldout_predictions.csv',index=False)
joblib.dump(dict(models=models,selected=selected,sklearn=version('scikit-learn'),training_bounds={c:[float(df.loc[df.eligible&(df.split=='train'),c].min()),float(df.loc[df.eligible&(df.split=='train'),c].max())] for c in sw.FEATURES}),ROOT/'models'/'retrieval.joblib')

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(12,4),constrained_layout=True)
for ax,split in zip(axs,['spatial_test','temporal_test','joint_test']):
    d=pred[(pred.model==selected)&(pred.split==split)]; ax.scatter(d.sm,d.prediction,s=7,alpha=.5); ax.plot([0,.65],[0,.65],'k--'); ax.set(xlim=(0,.65),ylim=(0,.65),xlabel='SMAP m³/m³',ylabel='Retrieved m³/m³',title=f'{split} (n={len(d)})')
fig.savefig(ROOT/'figures'/'04_heldout_scatter.png',dpi=150); plt.show()

**Checkpoint:** compare the selected model against train mean and seasonal ridge on every held-out split. A model can perform poorly or even have negative R². That is a valid finding. No hyperparameter tuning on the test years or spatial block.